## Load the Dataframe

In [1]:
import torch
import torch.nn as nn

if torch.cuda.is_available():
  device = 'cuda'
elif torch.backends.mps.is_available():
  device = 'mps'
else:
  device = 'cpu'

In [2]:
import pandas as pd

In [3]:
from google.colab import drive

drive.flush_and_unmount()
drive.mount("/content/drive", force_remount=True)

Drive not mounted, so nothing to flush and unmount.
Mounted at /content/drive


In [4]:
from pathlib import Path

DATA_DIR = Path("/content/drive/MyDrive/CFM_2024_Challenge")

print(DATA_DIR.exists())

True


In [5]:
path = Path("/content/drive/MyDrive/CFM_2024_Challenge/Xtrain.parquet")

df = pd.read_parquet(path)

## Categorical Features
Version 1: categorical features are one-hot encoded

In [6]:
import numpy as np

In [7]:
# For the moment, we will transform the following coordinates
# as one-hot
cat_columns = ["venue", "action", "side", "trade"]

for col in cat_columns:
    print(col, df[col].unique())

venue [4 1 5 2 0 3]
action ['A' 'D' 'U']
side ['A' 'B']
trade [False  True]


In [8]:
# get_dummies transforms the selected columns into one-hot
X = pd.get_dummies(
    df,
    columns=cat_columns,
    dtype=np.float32
)

In [9]:
# we remove obs_id and order_id for the moment
X.drop(columns=["obs_id", "order_id"], inplace=True)

In [10]:
## let's reduce the size of X

# Continuous variables: float64 to float32
float_cols = ["price", "bid", "ask"]
X[float_cols] = X[float_cols].astype("float32")

# Integer-valued features: use smaller integers
X["bid_size"] = X["bid_size"].astype("int32")
X["ask_size"] = X["ask_size"].astype("int32")
X["flux"] = X["flux"].astype("int32")

X.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16080000 entries, 0 to 16079999
Data columns (total 19 columns):
 #   Column       Dtype  
---  ------       -----  
 0   price        float32
 1   bid          float32
 2   ask          float32
 3   bid_size     int32  
 4   ask_size     int32  
 5   flux         int32  
 6   venue_0      float32
 7   venue_1      float32
 8   venue_2      float32
 9   venue_3      float32
 10  venue_4      float32
 11  venue_5      float32
 12  action_A     float32
 13  action_D     float32
 14  action_U     float32
 15  side_A       float32
 16  side_B       float32
 17  trade_False  float32
 18  trade_True   float32
dtypes: float32(16), int32(3)
memory usage: 1.1 GB


## Labels

In [11]:
y_df = pd.read_csv("/content/drive/MyDrive/CFM_2024_Challenge/y_train.csv")
y = torch.tensor(y_df["eqt_code_cat"].to_numpy(), dtype=torch.long)
del y_df

## Normalization


In [12]:
num_cols = ["price", "bid", "ask", "bid_size", "ask_size", "flux"]

mean = X[num_cols].mean()
std = X[num_cols].std()

X[num_cols] = (X[num_cols] - mean) / std




In [13]:
# convert X_scaled to a tensor
X_tensor = torch.tensor(X.to_numpy(), dtype=torch.float32)

### Free the memory

In [14]:
del X # we don't need the pd dataframe X anymore

import gc # garbage collector

gc.collect()


81

## Dataset Class

In [15]:
from torch.utils.data import Dataset, DataLoader

class StockDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y


    def __len__(self):
        return len(self.y) # e.g. how many indices in X

    # when the dataloader runs the loop i=0,...36

    def __getitem__(self, i):
        start = i * 100
        end = start + 100

        return self.X[start:end], self.y[i]

## DataLoaders

In [16]:
batch_size = 256

full_set = StockDataset(X_tensor, y)

del X_tensor, y # we do not need these in menory anymore
gc.collect()

full_loader = DataLoader(full_set, batch_size=batch_size, shuffle=True)

## Training Function

In [17]:

def train_full(model, optimizer, loss_fn, metric, train_loader,
               n_epochs, epoch_callback=None):

    history = {"train_losses": [], "train_metrics": []}

    for epoch in range(n_epochs):
        total_loss = 0.0
        metric.reset()
        model.train()

        if epoch_callback is not None:
            epoch_callback(model, epoch)

        for index, (X_batch, y_batch) in enumerate(train_loader):
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)

            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

            metric.update(y_pred, y_batch)
            train_metric = metric.compute().item()

            print(f"\rBatch {index + 1}/{len(train_loader)}", end="")
            print(f", loss={total_loss/(index+1):.4f}", end="")
            print(f", {train_metric=:.2%}", end="")

        history["train_losses"].append(total_loss / len(train_loader))
        history["train_metrics"].append(train_metric)

        print(f"\rEpoch {epoch + 1}/{n_epochs},                      "
              f"train loss: {history['train_losses'][-1]:.4f}, "
              f"train metric: {history['train_metrics'][-1]:.2%}")

    return history

# Model

In [18]:
from functools import partial
import torch.nn.functional as F

## Inception Block
To control the architecture a bit better, we separate the inception block from the permutation, pooling and classifier

In [19]:
# increase the number of output channels from the single branches
# n1 n2 n3 n4
#    n5 n6 n7
# This is the module without permutation, pooling and classifier
# WARNING: this block makes [B, 19, 100] -> [B, 64, 100] so before use the data has to
# be permuted, this block cannot be used alone
# after the block apply x = x.mean(dim=2) to bring the dimension to [B, 64]
class InceptionBlock(nn.Module):
  def __init__(self, in_channels, n1, n2, n3, n4, n5, n6, n7):
    super().__init__()
    Conv1d = partial(nn.Conv1d,
                            stride=1, padding="same", bias=False
                            )
    DefaultMaxpool = partial(nn.MaxPool1d,
                             stride=1, padding=n7//2)
    self.column1 = Conv1d(in_channels = in_channels,
                                 out_channels = 16,
                                 kernel_size=n1)
    self.column2 = nn.Sequential(
        Conv1d(in_channels = in_channels,
                      out_channels = 16,
                      kernel_size=n5),
        nn.ReLU(),
        Conv1d(in_channels = 16,
                      out_channels = 16,
                      kernel_size=n2)
    )
    self.column3 = nn.Sequential(
        Conv1d(in_channels = in_channels,
                      out_channels = 16,
                      kernel_size=n6),
        nn.ReLU(),
        Conv1d(in_channels = 16,
                      out_channels = 16,
                      kernel_size=n3)
    )
    self.column4 = nn.Sequential(
        DefaultMaxpool(kernel_size=n7),
        Conv1d(in_channels = in_channels,
                      out_channels = 16,
                      kernel_size=n4)
    )


  def forward(self, inputs):

    out1 = self.column1(inputs)
    out2 = self.column2(inputs)
    out3 = self.column3(inputs)
    out4 = self.column4(inputs)

    combined = torch.cat((out1, out2, out3, out4), dim=1)

    return F.relu(combined)

## BiGru Block

In [20]:
class GruBiBloc(nn.Module):
    def __init__(self, n_layers=2, input_dim=19,
                 hidden_dim=64, dropout=0.2):
        super().__init__()

        self.gru = nn.GRU(input_dim, hidden_dim, num_layers=n_layers,
            batch_first=True, dropout=dropout, bidirectional=True)

    def forward(self, X):
      # X: [B, 100, 19]

      _outputs, hidden_states = self.gru(X)
      # _outputs: [B, 100, 128]
      # hidden_states: [4, B, 64]

      # concatenate forward and backward hidden states
      n_dims = 2*self.gru.hidden_size # n_dims = 128

      top_states = hidden_states[-2:].permute(1, 0, 2).reshape(-1, n_dims)
      # top_states = hidden_states[-2:]
      # top_states: [2, B, 64] (extracts final layer)
      # top_states = top_states.permute(1, 0, 2)
      # top_states: [B, 2, 64]
      # top_states = top_states.reshape(-1, n_dims)
      # top_states: [B, 128] -> formula: [B, num_directions * hidden_dim]

      return top_states # [B, 128]

## BiGru + Inception

In [21]:
class BiGruInception(nn.Module):
  def __init__(self, in_channels, n1, n2, n3, n4, n5, n6, n7):
    super().__init__()

    self.inception1 = InceptionBlock(
        in_channels, n1, n2, n3, n4, n5, n6, n7
    )

    self.inception2 = InceptionBlock(
        64, n1, n2, n3, n4, n5, n6, n7
    )
    self.gru_block = GruBiBloc()

    self.linear_gru = nn.Linear(128, 64)
    self.linear_join1 = nn.Linear(128, 64)
    self.linear_join2 = nn.Linear(64, 32)
    self.linear_output = nn.Linear(32, 24)

  def forward(self, inputs): # inputs: [B,100, 19]
    p_inputs = inputs.permute(0, 2, 1)  # [B, 100, 19] -> [B, 19, 100]

    x1 = self.inception1(p_inputs)        # [B, 19, 100] -> [B, 64, 100]
    x1 = self.inception2(x1)             # [B, 64, 100] -> [B, 64, 100]

    x1 = x1.mean(dim=2)                  # [B, 64]

    x2 = self.gru_block(inputs) # [B, 100, 19] -> [B, 128]
    x2 = self.linear_gru(x2)       # [B, 128] -> [B, 64]

    combined = torch.cat((x1, x2), dim=1) # [B, 64]+[B, 64] ->[B, 128]
    x = F.relu(combined)

    x = self.linear_join1(x) # [B, 128] -> [B, 64]
    x = F.relu(x)

    x = self.linear_join2(x) # [B, 64] -> [B, 32]
    x = F.relu(x)

    return self.linear_output(x)       #[B,32] -> [B, 24]

## Training

In [22]:
%pip install -q torchmetrics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 53.6 MB/s eta 0:00:00


In [23]:
import torchmetrics

In [24]:
torch.manual_seed(42)

model_BiGru_Inception = BiGruInception(19, 1, 3, 5, 1, 1, 1, 3)
model_BiGru_Inception.to(device)



n_epochs = 71 # determined on train+val set!
xentropy = nn.CrossEntropyLoss()
optimizer = torch.optim.NAdam(model_BiGru_Inception.parameters())
accuracy = torchmetrics.Accuracy(task="multiclass",
                                 num_classes=24).to(device)

history = train_full(model_BiGru_Inception,optimizer,xentropy, accuracy, full_loader, n_epochs)


Epoch 1/71,                      train loss: 2.5163, train metric: 17.12%
Epoch 2/71,                      train loss: 2.1365, train metric: 29.51%
Epoch 3/71,                      train loss: 1.9158, train metric: 36.58%
Epoch 4/71,                      train loss: 1.7434, train metric: 42.08%
Epoch 5/71,                      train loss: 1.5935, train metric: 46.45%
Epoch 6/71,                      train loss: 1.4813, train metric: 50.17%
Epoch 7/71,                      train loss: 1.3854, train metric: 53.12%
Epoch 8/71,                      train loss: 1.3020, train metric: 55.77%
Epoch 9/71,                      train loss: 1.2316, train metric: 58.07%
Epoch 10/71,                      train loss: 1.1694, train metric: 60.10%
Epoch 11/71,                      train loss: 1.1206, train metric: 61.75%
Epoch 12/71,                      train loss: 1.0755, train metric: 63.13%
Epoch 13/71,                      train loss: 1.0385, train metric: 64.35%
Epoch 14/71,                      

In [25]:
torch.save(
    model_BiGru_Inception.state_dict(),
    "/content/drive/MyDrive/CFM_2024_Challenge/BiGru_Inception_1_full.pt"
)

# Testing

In [26]:
test_path = Path("/content/drive/MyDrive/CFM_2024_Challenge/Xtest.parquet")

df_test = pd.read_parquet(test_path)

# get_dummies transforms the selected columns into one-hot
X_test = pd.get_dummies(
    df_test,
    columns=cat_columns,
    dtype=np.float32
)

del df_test
gc.collect()

# we remove obs_id and order_id for the moment
X_test.drop(columns=["obs_id", "order_id"], inplace=True)

# scaling
X_test[num_cols] = (X_test[num_cols] - mean) / std

X_test_tensor = torch.tensor(X_test.to_numpy(), dtype=torch.float32)

del X_test
gc.collect()

14

In [27]:
class StockTestDataset(Dataset):
    def __init__(self, X):
        self.X = X

    def __len__(self):
        return len(self.X) // 100

    def __getitem__(self, index):
        start = index * 100
        end = start + 100
        return self.X[start:end]

In [28]:
test_set = StockTestDataset(X_test_tensor)
del X_test_tensor
gc.collect()

test_loader = DataLoader(test_set,batch_size=64,shuffle=False)

## Load the model

In [29]:
model_name = model_BiGru_Inception

## Run the model

In [30]:
# put the model in evaluation mode (e.g. disable dropout)
model_name.eval()

# list where we will store all predicted stock classes
predictions = []

# no gradients for predictions
with torch.no_grad():

    # take one batch from the test set
    for X_batch in test_loader:

        # move the batch to the same device as the model
        X_batch = X_batch.to(device)

        # for each observation, the model outputs 24 logits
        logits = model_name(X_batch)

        # choose the class with the largest logit
        preds = logits.argmax(dim=1)

        # move predictions back to CPU and add them to our list
        predictions.extend(preds.cpu().numpy())

In [31]:
submission = pd.DataFrame({
    "obs_id": np.arange(len(predictions)),
    "eqt_code_cat": predictions
})

submission_path = "/content/drive/MyDrive/CFM_2024_Challenge/predictions4.csv"

submission.to_csv(submission_path, index=False)


In [32]:
submission.head()

,obs_id,eqt_code_cat
0,0,13
1,1,3
2,2,19
3,3,1
4,4,22
